<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/Image_classifier_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np

In [22]:
def init_parameter(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) # Removed *0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [23]:
def output(W , b , X , L):
  A = X.T
  activation = [A]
  Zs = []
  def relu(z):
    return np.maximum(0,z)
  def sigmoid(z):
    return 1/(1+np.exp(-z))

  for l in range(1, L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)

    A = relu(Z)
    activation.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  Zs.append(ZL)

  AL = sigmoid(ZL)
  return activation , Zs , AL

In [24]:
def loss( y , AL ):
  m = y.shape[0]
  epsilon = 1e-8 # Small value to prevent log(0)
  temp2 = 0
  for i in range(len(y)):
    # Clip AL values to be within (epsilon, 1 - epsilon) to avoid log(0)
    AL_clipped = np.clip(AL[0, i], epsilon, 1 - epsilon)
    temp = y[i] * np.log(AL_clipped) + (1 - y[i])* np.log(1 - AL_clipped)
    temp2 += temp
  return -temp2/m

In [25]:

def backward(W, b, activations, Zs, AL, Y):

    m = Y.shape[0]
    Y = Y.reshape(1, m)

    L = len(W)

    dW = {}
    db = {}

    dZ = AL - Y

    dW[L] = (1/m) * np.dot(dZ, activations[-1].T)
    db[L] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    dA = np.dot(W[L].T, dZ)

    for l in reversed(range(1, L)):

        dZ = np.array(dA, copy=True)
        dZ[Zs[l-1] <= 0] = 0

        dW[l] = (1/m) * np.dot(dZ, activations[l-1].T)
        db[l] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        dA = np.dot(W[l].T, dZ)

    return dW, db

In [26]:
def gradient_descent(W , b , dW , db , lr):

  for l in W.keys():
    W[l] = W[l] - lr * dW[l]
    b[l] = b[l] - lr * db[l]
  return W , b

In [27]:
def model(dim):
  W ,b = init_parameter(dim)
  for i in range(10000):
    L = len(dim)
    activation , Zs , AL = output(W , b , X , L)
    cost = loss(y , AL)
    dW , db = backward(W , b , activation , Zs , AL , y)
    W , b = gradient_descent(W , b , dW , db , 0.5)
    if i % 1000 == 0:
      print(f"Iteration {i}: Cost = {cost}")
  return W , b

In [31]:
dim = [2 ,4, 1]
W, b = model(dim)

Iteration 0: Cost = 1.2230541563195247
Iteration 1000: Cost = 0.24747849713643194
Iteration 2000: Cost = 0.2461187557791255
Iteration 3000: Cost = 0.24552946569016162
Iteration 4000: Cost = 0.24530808777737864
Iteration 5000: Cost = 0.24516830027122613
Iteration 6000: Cost = 0.24504262945628053
Iteration 7000: Cost = 0.24495926135074378
Iteration 8000: Cost = 0.2448904121196537
Iteration 9000: Cost = 0.24483037061928323


In [32]:
activation , Zs , AL = output(W , b , X , len(dim))
pred = (AL > 0.5)

In [33]:
accuracy = np.mean(pred == y)
print("Model Accuracy: ",accuracy*100,"%")

Model Accuracy:  88.0 %


In [37]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 775M/775M [00:12<00:00, 63.1MB/s]



In [39]:
import zipfile

zip_path = "dog-and-cat-classification-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [117]:
import os

os.listdir("dataset")

['PetImages']

In [48]:
cats = len(os.listdir("dataset/PetImages/Cat"))
dogs = len(os.listdir("dataset/PetImages/Dog"))

print(cats)
print(dogs)

12499
12499


In [61]:
from PIL import Image

img = Image.open("dataset/PetImages/Cat/0.jpg")

In [55]:
img_array = np.array(img)

print(img_array.shape)

(375, 500, 3)


In [ ]:
img = img.resize((64,64))

img_array = np.array(img)
print(img_array.shape)
img_array = img_array/255
print(img_array)

In [111]:
X = []
y = []

In [112]:
for i in os.listdir("dataset/PetImages/Cat"):
  img = Image.open("dataset/PetImages/Cat/"+i)
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(0)

In [113]:
for i in os.listdir("dataset/PetImages/Dog"):
  img = Image.open("dataset/PetImages/Dog/"+i)
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(1)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [118]:
X = np.array(X)
Y = np.array(y)
print(X.shape)
print(y.shape)

(24998, 64, 64, 3)
(24998,)


In [119]:
perm = np.random.permutation(X.shape[0])

X = X[perm]
Y = Y[perm]

In [122]:
print(Y)
X = X / 255.0
print(X)

[1 1 1 ... 0 1 1]
[[[[1.20617259e-05 1.21823431e-05 1.21220345e-05]
   [1.21220345e-05 1.22426518e-05 1.21823431e-05]
   [1.22426518e-05 1.23029604e-05 1.23029604e-05]
   ...
   [1.21220345e-05 1.21220345e-05 1.23632690e-05]
   [1.16395655e-05 1.16395655e-05 1.17601827e-05]
   [1.20014173e-05 1.20014173e-05 1.20617259e-05]]

  [[1.21220345e-05 1.22426518e-05 1.21823431e-05]
   [1.21823431e-05 1.23029604e-05 1.22426518e-05]
   [1.22426518e-05 1.23632690e-05 1.23632690e-05]
   ...
   [1.21220345e-05 1.20617259e-05 1.23029604e-05]
   [1.16395655e-05 1.16395655e-05 1.17601827e-05]
   [1.20014173e-05 1.20014173e-05 1.20617259e-05]]

  [[1.20617259e-05 1.21823431e-05 1.21220345e-05]
   [1.21823431e-05 1.23029604e-05 1.22426518e-05]
   [1.23029604e-05 1.23632690e-05 1.23029604e-05]
   ...
   [1.20617259e-05 1.20014173e-05 1.22426518e-05]
   [1.16395655e-05 1.15792568e-05 1.17601827e-05]
   [1.20617259e-05 1.20014173e-05 1.21220345e-05]]

  ...

  [[9.34783756e-06 9.52876345e-06 9.83030659e-06